In [1]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [2]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC7596/UGC7596_1/UGC-7596_1_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC7596/UGC7596_1/UGC-7596_1_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     359   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        58   (2094, 964)   float32   


In [3]:
image_cut = image[77:177, 0:1890]
image_error_cut = image_error[77:177, 0:1890]

In [4]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.00189
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [5]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4, params5):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4) +
        sersic_1d(x_hr, params5) #+
        #sersic_1d(x_hr, params6) #+
        #sersic_1d(x_hr, params7)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [6]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC7596/PSF/Real_seeing_UGC7596.txt")
sigma = seeing[-1] / 2.35
x_axes = np.arange(0, 100, 1)

In [7]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, params5, a, b):
    func = model_convolved(x, params1, params2, params3, params4, params5)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, params5, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    magenta_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    choco_area = single_integral(x, params5, a, b)
    #purple_area = single_integral(x, params6, a, b)
    #pink_area = single_integral(x, params7, a, b)
    sum_area = blue_area + green_area + magenta_area + orange_area + choco_area# + purple_area #+ pink_area
    total_area = total_integral(x, params1, params2, params3, params4, params5, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (magenta_area*100)/sum_area, 
            (orange_area*100)/sum_area, (choco_area*100)/sum_area]#, (purple_area*100)/sum_area]#, (pink_area*100)/sum_area]     

## Halpha

In [8]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC7596/UGC7596_1/Fit/Halpha_fit.csv", index_col=0)

In [9]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)


data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.07396915326147181
0.2014081967692415
0.09126220165514369
0.10973707156528395
0.3098551477388577
0.3175419795028448
0.40854357572260946
0.4091445731648095
0.16532986158651966
0.16811405508782162
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       1.167746e-03  1.039055e-22  9.114502e-25  8.244566e-26   
Green      5.750451e-17  1.563747e-03  3.571278e-11  4.606521e-14   
Magenta    1.726176e-02  1.561306e+01  6.050333e+01  6.710052e+00   
Orange     1.374531e-13  2.808035e-01  1.481465e+01  7.472847e+01   
Chocolate  9.998157e+01  8.410457e+01  2.468202e+01  1.856148e+01   

                 Peak 5  
Blue       5.056567e-22  
Green      1.043539e-04  
Magenta    7.105505e+00  
Orange     3.264523e-02  
Chocolate  9.286175e+01  


In [10]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                       fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

0.2218857245751202
0.5530432168944297
0.2756724648937593
0.3222120902501332
0.9039469373402542
0.9224204617009988
1.1889871847692013
1.1904823820385089
0.3322267090705454
0.3460472725859488
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       1.026549e-03  1.117952e-22  9.878414e-25  8.855933e-26   
Green      6.194282e-17  1.332341e-03  1.063855e-10  5.990893e-14   
Magenta    1.754983e-02  1.613607e+01  5.828406e+01  7.245419e+00   
Orange     2.272132e-13  3.348123e-01  1.633575e+01  7.362248e+01   
Chocolate  9.998142e+01  8.352779e+01  2.538019e+01  1.913211e+01   

                 Peak 5  
Blue       6.093566e-22  
Green      2.601511e-04  
Magenta    7.529705e+00  
Orange     4.764536e-02  
Chocolate  9.242239e+01  


In [11]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

0.3697130423618119
0.7424779046456146
0.468158241418253
0.5204526028255296
1.420269556753359
1.4387513232649891
1.8502530431232858
1.8519496774732223
0.5025863922546145
0.5369396538902877
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       7.088664e-04  1.353667e-22  1.204501e-24  1.058371e-25   
Green      7.439223e-17  9.520995e-04  1.633012e-08  1.150577e-13   
Magenta    1.831841e-02  1.754014e+01  5.274239e+01  8.809712e+00   
Orange     5.743098e-13  4.877524e-01  2.033884e+01  7.070360e+01   
Chocolate  9.998097e+01  8.197116e+01  2.691877e+01  2.048669e+01   

                 Peak 5  
Blue       8.179379e-22  
Green      4.754806e-04  
Magenta    8.280679e+00  
Orange     8.005702e-02  
Chocolate  9.163879e+01  


## HBeta

In [12]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC7596/UGC7596_1/Fit/Hbeta_fit.csv", index_col = 0)

In [13]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)


data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.17445555899164616
0.19179004745616474
0.09566388095503581
0.09567828007667376
0.1392618102098992
0.13915622899190638
0.146250732420725
0.14622655213746355
0.07998073822025531
0.07986338325845009
                 Peak 1        Peak 2     Peak 3        Peak 4        Peak 5
Blue       6.555321e+01  4.727854e-03   0.000292  8.353306e-05  1.387493e-02
Green      0.000000e+00  3.462333e+01   0.000131  5.777934e-45  2.181771e+01
Magenta    0.000000e+00  3.240729e-02  50.900213  1.475215e+01  1.711339e-24
Orange     8.257005e-70  5.264944e-07   4.610259  4.348800e+01  4.649931e-11
Chocolate  3.444679e+01  6.533953e+01  44.489104  4.175977e+01  7.816842e+01


In [14]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                       fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

0.48842208702036416
0.5314607195280192
0.2853541087118239
0.28537832698337895
0.420477016072799
0.42017177474216777
0.43776293217130857
0.4378653054033619
0.23950690387331747
0.23918712045499538
                 Peak 1     Peak 2     Peak 3        Peak 4        Peak 5
Blue       6.309243e+01   0.004861   0.000295  8.490684e-05  1.425477e-02
Green      0.000000e+00  34.133103   0.001029  6.070742e-29  2.167623e+01
Magenta    0.000000e+00   0.149292  50.509764  1.600637e+01  2.860812e-08
Orange     5.721376e-66   0.000002   5.287827  4.214323e+01  4.147196e-10
Chocolate  3.690757e+01  65.712742  44.201085  4.185032e+01  7.830952e+01


In [15]:
ddata1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')


0.7059520593626414
0.7516003075155238
0.4682876182887066
0.46825258052090807
0.7104884086447392
0.7100633507360365
0.8203714165115947
0.8210327427808104
0.39887368814632274
0.3984770913689989
                 Peak 1     Peak 2     Peak 3        Peak 4        Peak 5
Blue       6.309243e+01   0.005237   0.000304  8.588577e-05  1.526377e-02
Green      0.000000e+00  32.521641   0.010281  2.668898e-18  2.161815e+01
Magenta    0.000000e+00   0.740264  49.474887  1.615681e+01  1.769255e-04
Orange     5.721376e-66   0.000017   6.924959  3.928357e+01  5.997327e-09
Chocolate  3.690757e+01  66.732840  43.589569  4.455954e+01  7.836641e+01


## NII

In [16]:
mask_NII = (wv > l_NII_2-8)*(wv < l_NII_2+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC7596/UGC7596_1/Fit/NII_fit.csv", index_col = 0)

In [17]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)


data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.20087245604466733
0.19329628390511705
0.08596309287239383
0.0858568501950818
0.11175607079484912
0.11185620275714467
0.11937092432646715
0.1157091423905449
0.07184667245104134
0.07178679643616395
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5
Blue       7.697489e+01   0.950917   0.326958   0.181685   1.609956
Green      1.689526e-01  39.256685   9.531461   2.590950  32.217536
Magenta    1.006079e-09   5.425463  47.260057   9.012990   1.105582
Orange     2.702658e-13   0.001170   1.255863  49.522140   0.000089
Chocolate  2.285616e+01  54.365764  41.625661  38.692235  65.066836


In [18]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                       fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

0.565020254761239
0.543451286607683
0.2565140102225713
0.25622008853423617
0.332269795049948
0.332506660262098
0.45680178561181933
0.44565048493397946
0.21607009221821483
0.21589260973912572
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5
Blue       7.544083e+01   0.959648   0.330754   0.198824   1.613352
Green      1.826503e-01  38.605177   9.698226   3.085466  32.266842
Magenta    1.600208e-09   5.777666  46.424589  12.500627   1.213867
Orange     3.595769e-13   0.001452   1.546055  43.743873   0.000110
Chocolate  2.437651e+01  54.656057  42.000376  40.471211  64.905827


In [19]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

0.8014494813636953
0.7748005333144241
0.42291339465363675
0.4224755637504103
0.542715927414908
0.542883506910081
0.6313142640642992
0.6194924585762931
0.36221315993769193
0.36191653849487015
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5
Blue       7.113703e+01   0.980084   0.339831   0.217298   1.623914
Green      2.222802e-01  37.073361  10.117511   3.467945  32.344330
Magenta    4.049439e-09   6.695587  44.255878  14.801518   1.505077
Orange     6.814070e-13   0.002319   2.432426  37.590613   0.000174
Chocolate  2.864069e+01  55.248649  42.854354  43.922627  64.526505


## SII

In [20]:
mask_SII = (wv > l_SII_1-8)*(wv < l_SII_1+8)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC7596/UGC7596_1/Fit/SII_fit.csv", index_col = 0)

In [21]:
fit

,Component 1,Component 2,Component 3,Component 4,Component 5
I_e,0.0061,0.0200,0.0423,0.6593,0.0215
r_e,7.5345,1.1527,5.2142,0.0156,118.3076
n,2.6881,0.3398,0.4355,4.1944,0.7676
x_0,38.1988,61.6166,70.2574,75.0175,58.8095


In [22]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)


data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.20022717385508707
0.21095082731860204
0.10702089513733576
0.10697984668694527
0.14198807723033258
0.14198748655770513
0.275708773928192
0.14035746540079064
0.07812427822092137
0.07800238950413921
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       6.789824e+01  4.139344e-01  1.151861e-01  3.676691e-02   
Green      0.000000e+00  2.048133e+01  3.778040e-12  4.858370e-85   
Magenta    3.038464e-13  1.182502e+01  5.164888e+01  1.523525e+01   
Orange     6.328149e-17  7.845851e-12  3.015481e-02  6.076128e+01   
Chocolate  3.210176e+01  6.727971e+01  4.820578e+01  2.396670e+01   

                 Peak 5  
Blue       8.343991e-01  
Green      3.293381e+00  
Magenta    3.090367e+00  
Orange     1.340174e-12  
Chocolate  9.278185e+01  


In [23]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                       fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

0.5666784921228334
0.5922648825194468
0.31614202224023896
0.315982137473084
0.4251655243835776
0.4242980532869862
0.9503953563316679
0.5467404261836566
0.23781975398936178
0.23747831329689967
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       6.597248e+01  4.223162e-01  1.157468e-01  4.491462e-02   
Green      0.000000e+00  1.884075e+01  3.457213e-08  1.247479e-44   
Magenta    1.007054e-12  1.242062e+01  5.131549e+01  1.963160e+01   
Orange     6.918136e-17  9.366188e-12  2.739805e-01  5.241118e+01   
Chocolate  3.402752e+01  6.831632e+01  4.829478e+01  2.791230e+01   

                 Peak 5  
Blue       8.268727e-01  
Green      4.460953e+00  
Magenta    3.326546e+00  
Orange     1.479252e-12  
Chocolate  9.138563e+01  


In [24]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

0.8195339081759802
0.8457517933452776
0.5109141406310894
0.5105572367943682
0.7134615680976897
0.701203299877303
1.203016193610497
0.7757547752123765
0.40869566802094526
0.4081808257023852
                 Peak 1        Peak 2     Peak 3        Peak 4        Peak 5
Blue       6.078676e+01  4.409742e-01   0.115885  5.367444e-02  8.140218e-01
Green      0.000000e+00  1.524063e+01   0.000009  9.424677e-31  6.734458e+00
Magenta    6.260115e-12  1.389804e+01  49.816501  2.303787e+01  3.942362e+00
Orange     8.626535e-17  1.415296e-11   2.105435  4.383426e+01  1.886028e-12
Chocolate  3.921324e+01  7.042035e+01  47.962170  3.307419e+01  8.850916e+01


## OIII

In [25]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC7596/UGC7596_1/Fit/OIII_fit.csv", index_col = 0)

In [26]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)


data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.21735869841033445
0.2220322171256393
0.1162625893464257
0.11638734180612878
0.16700875654855543
0.1676030468696077
0.17256009735481076
0.13775589820352
0.09785103870595513
0.09777297579312873
              Peak 1     Peak 2        Peak 3     Peak 4     Peak 5
Blue       68.631798   0.831561  2.530289e-01   0.143316   1.412407
Green       0.086419  27.039188  4.249512e+00   1.019345  20.564750
Magenta     0.000833  10.323960  5.320237e+01  13.618216   4.472250
Orange      0.000000   0.000000  6.079559e-67  45.167504   0.000000
Chocolate  31.280949  61.805291  4.229509e+01  40.051619  73.550593


In [27]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                       fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

0.6138769106090082
0.624987111870504
0.3469431425090016
0.3472151246816707
0.4926223250209314
0.4940961596147786
0.4853904323558255
0.3953296867682528
0.2944015962015202
0.2941955044959128
              Peak 1     Peak 2        Peak 3     Peak 4     Peak 5
Blue       66.679694   0.839333  2.580387e-01   0.153176   1.415153
Green       0.093170  26.394049  4.375707e+00   1.101910  20.665699
Magenta     0.000926  10.637641  5.235173e+01  14.822550   4.589270
Orange      0.000000   0.000000  1.336597e-12  41.208115   0.000000
Chocolate  33.226210  62.128977  4.301452e+01  42.714249  73.329877


In [28]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

0.9061584155901994
0.9154572140697784
0.5720036746936831
0.5721580451197062
0.789498618337551
0.7910846855687265
0.7264968358905002
0.6133507958056349
0.49391544528126774
0.4936910736469293
              Peak 1     Peak 2     Peak 3     Peak 4     Peak 5
Blue       62.378057   0.857689   0.270296   0.171547   1.424318
Green       0.109323  24.902492   4.702634   1.271959  20.846923
Magenta     0.001168  11.447284  50.297981  17.404067   4.903014
Orange      0.000000   0.000000   0.002103  33.593545   0.000000
Chocolate  37.511452  62.792535  44.726987  47.558882  72.825745
